# CARD vs FiHaRD vs STAR-Hand: aligned probabilistic forecasts

This notebook makes one publication-ready figure from the aligned `pick_up_screwd` prediction caches. It shows only hand skeletons in a 4 × 5 grid: one ground-truth-only row followed by one row per model, at five forecast moments. Model panels overlay all 10 sampled futures and the oracle best-of-10 sample.

Important: **oracle best-of-10 uses the ground truth for selection** and should be described that way in the caption. AssemblyHands cache coordinates are wrist-relative at every frame, so global wrist displacement is not present. Every skeleton is explicitly re-rooted to its wrist before plotting.

In [ ]:
from __future__ import annotations

import gc
import importlib
import json
import sys
from dataclasses import fields
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import torch
import yaml

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'configs' / 'models').exists():
    REPO_ROOT = Path('/home/fagnelli/diffusion_hands')
assert (REPO_ROOT / 'configs' / 'models').exists(), REPO_ROOT

VENDOR_SPLINE = REPO_ROOT / 'vendor' / 'splineeqnet'
for path in (REPO_ROOT, VENDOR_SPLINE):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from common.dataset_graphs import resolve_local_hand_graph_metadata

MODEL_NAMES = ('card', 'fihard', 'star_hand')
CACHE_MODEL_NAMES = (*MODEL_NAMES, 'comusion')
MODEL_TITLES = {'card': 'CARD', 'fihard': 'FiHaRD', 'star_hand': 'STAR-Hand', 'comusion': 'CoMusion'}
MODEL_COLORS = {'card': '#E76F51', 'fihard': '#8E6BBE', 'star_hand': '#D99A22', 'comusion': '#55B7A4'}
MODEL_DARK = {'card': '#9D2F17', 'fihard': '#4E2A78', 'star_hand': '#8A5900', 'comusion': '#126B5C'}
CACHE_CANDIDATES = (
    REPO_ROOT / 'out/bayesian_signed_rank/prediction_cache',
    REPO_ROOT / 'out/bayesian_signed_rank/assembly_pick_up_screwd_card_variants_vs_comusion/prediction_cache',
)
CACHE_DIR = next(
    (directory for directory in CACHE_CANDIDATES if all((directory / f'{name}_predictions.pt').exists() for name in CACHE_MODEL_NAMES)),
    None,
)
if CACHE_DIR is None:
    raise FileNotFoundError(f'No complete aligned prediction cache found in: {CACHE_CANDIDATES}')
OUTPUT_DIR = REPO_ROOT / 'results' / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set six explicit indices to force particular aligned examples. None uses the selection rule below.
SAMPLE_INDICES = None
PROJECTION = (0, 1)  # (0, 1)=XY; try (0, 2)=XZ or (1, 2)=YZ
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Repository: {REPO_ROOT}\nCentre inference device: {DEVICE}')

## Separate competitor comparison: STAR-Hand vs CoMusion

CoMusion is the only external competitor with a complete ten-candidate cache aligned to all 4,981 test windows. This section compares it with STAR-Hand, the best of CARD, FiHaRD, and STAR-Hand on the selected moving subset. It selects six new moving sequences, excluding the examples used above and nearby dataset windows.

In [ ]:
COMPETITOR_SECTION_CODE = r'''
COMPETITOR_NAMES = ('star_hand', 'comusion')

def select_competitor_examples(gt, caches, excluded_indices, count=6):
    rooted = gt - gt[:, :, WRIST : WRIST + 1, :]
    visible = rooted[..., list(PROJECTION)]
    endpoint_motion = torch.linalg.vector_norm(visible[:, -1] - visible[:, 0], dim=-1).mean(dim=1)
    temporal_spread = torch.linalg.vector_norm(visible - visible[:, 0:1], dim=-1).mean(dim=(1, 2))
    endpoint_threshold = torch.quantile(endpoint_motion, 0.75)
    spread_threshold = torch.quantile(temporal_spread, 0.75)

    model_errors = []
    batch_size = 256
    for name in COMPETITOR_NAMES:
        chunks = []
        predictions = caches[name]['pred_candidates']
        for start in range(0, gt.shape[0], batch_size):
            stop = min(start + batch_size, gt.shape[0])
            error = torch.linalg.vector_norm(
                predictions[start:stop] - gt[start:stop, None], dim=-1
            ).mean(dim=(2, 3))
            chunks.append(error.min(dim=1).values)
        model_errors.append(torch.cat(chunks))
    best_error = torch.stack(model_errors).min(dim=0).values
    eligible = torch.nonzero(
        (endpoint_motion >= endpoint_threshold) & (temporal_spread >= spread_threshold),
        as_tuple=False,
    ).flatten()
    ranked = eligible[torch.argsort(best_error[eligible])]

    selected = []
    for index_tensor in ranked:
        index = int(index_tensor)
        if any(abs(index - old) < 50 for old in excluded_indices):
            continue
        if any(abs(index - previous) < 50 for previous in selected):
            continue
        selected.append(index)
        if len(selected) == count:
            break
    if len(selected) != count:
        raise RuntimeError(f'Found only {len(selected)} distinct competitor examples')
    return selected, float(endpoint_threshold), float(spread_threshold)

competitor_indices, competitor_endpoint_threshold, competitor_spread_threshold = select_competitor_examples(
    reference['gt_future'], caches, sample_indices
)
for name in COMPETITOR_NAMES:
    assert torch.equal(caches[name]['context'][competitor_indices], reference['context'][competitor_indices])
    assert torch.equal(caches[name]['gt_future'][competitor_indices], reference['gt_future'][competitor_indices])

competitor_target = reference['gt_future'][competitor_indices].numpy()
competitor_candidates = {
    name: caches[name]['pred_candidates'][competitor_indices].numpy() for name in COMPETITOR_NAMES
}
competitor_oracle_index = {}
competitor_oracle_mpjpe = {}
for name in COMPETITOR_NAMES:
    errors = np.linalg.norm(
        competitor_candidates[name] - competitor_target[:, None], axis=-1
    ).mean(axis=(2, 3))
    competitor_oracle_index[name] = errors.argmin(axis=1)
    competitor_oracle_mpjpe[name] = errors.min(axis=1)
competitor_winners = np.argmin(
    np.stack([competitor_oracle_mpjpe[name] for name in COMPETITOR_NAMES]), axis=0
)
print('Competitor samples:', competitor_indices)
print('Visible-motion thresholds:', competitor_endpoint_threshold, competitor_spread_threshold)
print('Oracle MPJPE:', {name: np.round(values, 4).tolist() for name, values in competitor_oracle_mpjpe.items()})
print('Best model:', [MODEL_TITLES[COMPETITOR_NAMES[index]] for index in competitor_winners])

def make_competitor_figure(position):
    target_i = competitor_target[position]
    candidates_i = {name: competitor_candidates[name][position] for name in COMPETITOR_NAMES}
    limits = {'ground_truth': row_limits([target_i[frame] for frame in MOMENTS])}
    for name in COMPETITOR_NAMES:
        limits[name] = row_limits([pose for frame in MOMENTS for pose in candidates_i[name][:, frame]])

    fig, axes = plt.subplots(3, 5, figsize=(15.5, 8.3))
    fig.subplots_adjust(left=0.15, right=0.99, bottom=0.04, top=0.83, hspace=0.08, wspace=0.035)
    gt_lo, gt_hi = limits['ground_truth']
    for column, frame in enumerate(MOMENTS):
        ax = axes[0, column]
        plot_skeleton(ax, target_i[frame], GT_COLOR, lw=2.15, alpha=1.0, zorder=4)
        ax.set_xlim(gt_lo[0], gt_hi[0])
        ax.set_ylim(gt_lo[1], gt_hi[1])
        ax.set_aspect('equal', adjustable='box')
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_title(f'$t={frame + 1}$', fontsize=22, pad=8)
        if column == 0:
            ax.set_ylabel('Ground truth', fontsize=24, fontweight='semibold', rotation=0, labelpad=82, va='center')

    for model_row, name in enumerate(COMPETITOR_NAMES, start=1):
        pose_lo, pose_hi = limits[name]
        for column, frame in enumerate(MOMENTS):
            ax = axes[model_row, column]
            for sampled_pose in candidates_i[name][:, frame]:
                plot_skeleton(ax, sampled_pose, MODEL_COLORS[name], lw=0.75, alpha=0.18, zorder=1)
            best_pose = candidates_i[name][competitor_oracle_index[name][position], frame]
            plot_skeleton(ax, best_pose, MODEL_DARK[name], lw=2.15, alpha=0.98, zorder=5)
            ax.set_xlim(pose_lo[0], pose_hi[0])
            ax.set_ylim(pose_lo[1], pose_hi[1])
            ax.set_aspect('equal', adjustable='box')
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            if column == 0:
                ax.set_ylabel(MODEL_TITLES[name], fontsize=24, fontweight='semibold', rotation=0, labelpad=72, va='center')

    fig.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 0.975), ncol=3, frameon=False, fontsize=18)
    test_index = competitor_indices[position]
    stem = OUTPUT_DIR / f'competitors_star_hand_example_{position + 1:02d}_index_{test_index}'
    fig.savefig(stem.with_suffix('.pdf'), bbox_inches='tight')
    fig.savefig(stem.with_suffix('.png'), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    return stem

competitor_output_stems = [make_competitor_figure(position) for position in range(6)]
for stem in competitor_output_stems:
    print(f'Saved {stem.with_suffix(".pdf")}')
    print(f'Saved {stem.with_suffix(".png")}')
'''

## Load the aligned caches

`mmap=True` prevents three approximately 479 MB cache files from being copied fully into RAM. Only the selected sequence is converted to NumPy.

In [ ]:
def load_cache(path: Path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False, mmap=True)
    except TypeError:  # older PyTorch without mmap
        return torch.load(path, map_location='cpu', weights_only=False)

caches = {name: load_cache(CACHE_DIR / f'{name}_predictions.pt') for name in CACHE_MODEL_NAMES}
reference = caches['card']
expected_shapes = {
    'pred_candidates': (4981, 10, 30, 21, 3),
    'gt_future': (4981, 30, 21, 3),
    'context': (4981, 70, 21, 3),
}
for name, cache in caches.items():
    for key, shape in expected_shapes.items():
        assert tuple(cache[key].shape) == shape, (name, key, cache[key].shape)
    meta = cache['metadata']
    assert meta['dataset'] == 'assembly' and meta['action_filter'] == 'pick_up_screwd'
    print(name, {key: tuple(cache[key].shape) for key in expected_shapes})

graph = resolve_local_hand_graph_metadata('assembly', (5, 26))
WRIST = int(graph['wrist_index'])
LINKS = tuple(graph['links'])
FINGERTIPS = (0, 1, 2, 3, 4)  # terminal nodes in the AssemblyHands local topology
print(f'Local wrist index: {WRIST}; skeleton edges: {len(LINKS)}')

## Select six moving, well-predicted sequences

The default selector requires visibly substantial motion in the plotted projection: both first-to-final root-relative joint displacement and average displacement from the initial pose must exceed their 75th-percentile test-set thresholds. Among those sequences, it ranks examples by the lowest oracle best-of-10 MPJPE achieved by CARD, FiHaRD, or STAR-Hand, then greedily keeps six examples separated by at least 50 dataset positions to avoid near-duplicate windows. Set `SAMPLE_INDICES` above to six predetermined indices. Because this is a deliberately favorable qualitative selection, report the selection rule with the figure.

In [ ]:
def choose_sequences(gt: torch.Tensor, caches, sample_indices=None) -> list[int]:
    if sample_indices is not None:
        indices = [int(index) for index in sample_indices]
        if len(indices) != 6 or len(set(indices)) != 6:
            raise ValueError('SAMPLE_INDICES must contain six distinct indices')
        if any(not 0 <= index < gt.shape[0] for index in indices):
            raise IndexError(f'SAMPLE_INDICES must be inside [0, {gt.shape[0] - 1}]')
        return indices
    rooted = gt - gt[:, :, WRIST : WRIST + 1, :]
    visible = rooted[..., list(PROJECTION)]
    endpoint_motion = torch.linalg.vector_norm(visible[:, -1] - visible[:, 0], dim=-1).mean(dim=1)
    temporal_spread = torch.linalg.vector_norm(visible - visible[:, 0:1], dim=-1).mean(dim=(1, 2))
    endpoint_threshold = torch.quantile(endpoint_motion, 0.75)
    spread_threshold = torch.quantile(temporal_spread, 0.75)

    # Compute each model's oracle best-of-10 MPJPE in small batches.
    model_errors = []
    batch_size = 256
    for name in MODEL_NAMES:
        chunks = []
        predictions = caches[name]['pred_candidates']
        for start in range(0, gt.shape[0], batch_size):
            stop = min(start + batch_size, gt.shape[0])
            distance = torch.linalg.vector_norm(
                predictions[start:stop] - gt[start:stop, None], dim=-1
            ).mean(dim=(2, 3))
            chunks.append(distance.min(dim=1).values)
        model_errors.append(torch.cat(chunks))
    best_any_model = torch.stack(model_errors).min(dim=0).values
    eligible = torch.nonzero(
        (endpoint_motion >= endpoint_threshold) & (temporal_spread >= spread_threshold),
        as_tuple=False,
    ).flatten()
    ranked = eligible[torch.argsort(best_any_model[eligible])]

    indices = []
    min_index_separation = 50
    for index_tensor in ranked:
        index = int(index_tensor)
        if all(abs(index - previous) >= min_index_separation for previous in indices):
            indices.append(index)
        if len(indices) == 6:
            break
    if len(indices) < 6:
        raise RuntimeError('Could not find six separated moving sequences')
    print(
        f'Visible-motion thresholds (75th percentiles): '
        f'endpoint={float(endpoint_threshold):.6f}, spread={float(spread_threshold):.6f}'
    )
    return indices

sample_indices = choose_sequences(reference['gt_future'], caches, SAMPLE_INDICES)
for name in MODEL_NAMES[1:]:
    # Validate alignment where it matters without scanning/copying every tensor.
    assert torch.equal(caches[name]['context'][sample_indices], reference['context'][sample_indices])
    assert torch.equal(caches[name]['gt_future'][sample_indices], reference['gt_future'][sample_indices])

context = reference['context'][sample_indices].numpy()
target = reference['gt_future'][sample_indices].numpy()
candidates = {name: caches[name]['pred_candidates'][sample_indices].numpy() for name in MODEL_NAMES}
oracle_index = {}
oracle_mpjpe = {}
for name in MODEL_NAMES:
    per_candidate = np.linalg.norm(candidates[name] - target[:, None], axis=-1).mean(axis=(2, 3))
    oracle_index[name] = per_candidate.argmin(axis=1)
    oracle_mpjpe[name] = per_candidate.min(axis=1)
print('Selected samples:', sample_indices)
print('Oracle indices:', {k: v.tolist() for k, v in oracle_index.items()})
print('Oracle MPJPE:', {k: np.round(v, 4).tolist() for k, v in oracle_mpjpe.items()})
winner_ids = np.argmin(np.stack([oracle_mpjpe[name] for name in MODEL_NAMES]), axis=0)
print('Best qualifying model:', [MODEL_TITLES[MODEL_NAMES[index]] for index in winner_ids])

## Deterministic centre disabled

The figure intentionally omits deterministic conditional centres and shows only stochastic samples plus the oracle best-of-10 sample.

In [ ]:
# Kept as an explicit switch point for future variants of the figure.
centres = None

## Plot five hand snapshots per model and export

The first row contains ground truth only; the remaining rows contain CARD, FiHaRD, and STAR-Hand without a ground-truth overlay. Columns are five evenly spaced forecast moments. Limits are shared within each row so that the hands remain large while temporal scale stays consistent for each source.

In [ ]:
GT_COLOR = '#2CA02C'
MOMENTS = tuple(np.linspace(0, target.shape[1] - 1, 5, dtype=int))

def projected(points):
    return np.asarray(points)[..., list(PROJECTION)]

def row_limits(poses, padding=0.06):
    points = np.concatenate([projected(pose - pose[WRIST]).reshape(-1, 2) for pose in poses], axis=0)
    lo = points.min(axis=0)
    hi = points.max(axis=0)
    span = np.maximum(hi - lo, 1e-6)
    return lo - padding * span, hi + padding * span

def plot_skeleton(ax, pose, color, lw=1.2, alpha=1.0, zorder=2):
    xy = projected(pose - pose[WRIST])
    for a, b in LINKS:
        ax.plot(xy[[a, b], 0], xy[[a, b], 1], color=color, lw=lw, alpha=alpha, zorder=zorder)
    ax.scatter(xy[:, 0], xy[:, 1], s=4.5, color=color, alpha=alpha, zorder=zorder + 1, linewidths=0)

plt.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 24,
    'axes.labelsize': 18,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})
legend_handles = [
    Line2D([0], [0], color=GT_COLOR, lw=2.0, label='Ground truth'),
    Line2D([0], [0], color='#999999', lw=1.0, alpha=0.5, label='Sampled hands (K=10)'),
    Line2D([0], [0], color='#444444', lw=2.1, label='Oracle best-of-10'),
]
def make_figure(example_position: int):
    target_i = target[example_position]
    candidates_i = {name: candidates[name][example_position] for name in MODEL_NAMES}
    limits = {'ground_truth': row_limits([target_i[frame] for frame in MOMENTS])}
    for name in MODEL_NAMES:
        row_poses = []
        for frame in MOMENTS:
            row_poses.extend(candidates_i[name][:, frame])
        limits[name] = row_limits(row_poses)

    fig, axes = plt.subplots(4, 5, figsize=(15.5, 10.4))
    fig.subplots_adjust(left=0.15, right=0.99, bottom=0.035, top=0.85, hspace=0.08, wspace=0.035)
    gt_lo, gt_hi = limits['ground_truth']
    for column, frame in enumerate(MOMENTS):
        ax = axes[0, column]
        plot_skeleton(ax, target_i[frame], GT_COLOR, lw=2.15, alpha=1.0, zorder=4)
        ax.set_xlim(gt_lo[0], gt_hi[0])
        ax.set_ylim(gt_lo[1], gt_hi[1])
        ax.set_aspect('equal', adjustable='box')
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_title(f'$t={frame + 1}$', fontsize=22, pad=8)
        if column == 0:
            ax.set_ylabel('Ground truth', fontsize=24, fontweight='semibold', rotation=0, labelpad=82, va='center')

    for model_row, name in enumerate(MODEL_NAMES, start=1):
        pose_lo, pose_hi = limits[name]
        for column, frame in enumerate(MOMENTS):
            ax = axes[model_row, column]
            for sampled_pose in candidates_i[name][:, frame]:
                plot_skeleton(ax, sampled_pose, MODEL_COLORS[name], lw=0.75, alpha=0.18, zorder=1)
            best_pose = candidates_i[name][oracle_index[name][example_position], frame]
            plot_skeleton(ax, best_pose, MODEL_DARK[name], lw=2.15, alpha=0.98, zorder=5)
            ax.set_xlim(pose_lo[0], pose_hi[0])
            ax.set_ylim(pose_lo[1], pose_hi[1])
            ax.set_aspect('equal', adjustable='box')
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            if column == 0:
                ax.set_ylabel(MODEL_TITLES[name], fontsize=24, fontweight='semibold', rotation=0, labelpad=72, va='center')

    fig.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 0.975), ncol=3, frameon=False, fontsize=18)
    sample_index = sample_indices[example_position]
    stem = OUTPUT_DIR / f'card_fihard_star_hand_example_{example_position + 1:02d}_index_{sample_index}'
    fig.savefig(stem.with_suffix('.pdf'), bbox_inches='tight')
    fig.savefig(stem.with_suffix('.png'), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    return stem

output_stems = [make_figure(position) for position in range(len(sample_indices))]
for stem in output_stems:
    print(f'Saved {stem.with_suffix(".pdf")}')
    print(f'Saved {stem.with_suffix(".png")}')

In [ ]:
exec(COMPETITOR_SECTION_CODE)

### Suggested caption

**Qualitative comparison on selected moving AssemblyHands test sequences.** Examples exceed the 75th-percentile thresholds for both first-to-final joint displacement and temporal pose spread in the plotted projection, and are selected by the lowest oracle best-of-10 MPJPE achieved by at least one compared model, with separation between dataset windows. Each exported figure contains one sequence. The first row shows ground truth only; the remaining rows show CARD, FiHaRD, and STAR-Hand without ground-truth overlays. Columns show five evenly spaced forecast moments. Light model-coloured skeletons denote ten sampled futures, and the darker model-coloured skeleton denotes the oracle best-of-10 prediction selected by minimum future MPJPE. Every hand is translated to its wrist because the evaluation representation does not retain global wrist displacement. Limits are shared within rows.